# Working with CSV files: `pd.read_csv` and its parameters

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
Data files are in `data/`: a 1,000-row sample of a job-seekers dataset (`aug_train.csv`), a tab-separated movie list,
IPL matches 2008-2020, the first rows of a Zomato restaurants file, and two tiny files for the `header` and bad-lines examples.

## 1. Import pandas

In [ ]:
import pandas as pd

pd.__version__   # this Notebook was written for pandas 3.x

## 2. Open a CSV file on our machine

In [ ]:
# the path is relative to this Notebook's folder
df = pd.read_csv("data/aug_train.csv")
print(df.shape)   # (rows, columns)
df.head()

## 3. Open a CSV file from a URL (needs internet)

In [ ]:
url = "https://raw.githubusercontent.com/cs109/2014_data/master/countries.csv"

# pandas downloads the file itself
countries = pd.read_csv(url)
countries.head()

In [ ]:
# Some servers refuse requests that do not look like a browser.
# Then we send a User-Agent header through storage_options.
headers = {"User-Agent": "Mozilla/5.0"}
pd.read_csv(url, storage_options=headers).shape

## 4. `sep` and `names`: a tab-separated file without column names

In [ ]:
# Before: the default separator is a comma, so each whole line lands in ONE column
pd.read_csv("data/movie_titles_metadata.tsv").head(3)

In [ ]:
# After: sep="\t" splits on tabs; names= gives the columns their names
cols = ["sno", "name", "release_year", "rating", "votes", "genres"]
movies = pd.read_csv("data/movie_titles_metadata.tsv", sep="\t", names=cols)
movies.head(3)

In [ ]:
# without names=, the first movie would have become the header: 616 rows instead of 617
print(pd.read_csv("data/movie_titles_metadata.tsv", sep="\t").shape, movies.shape)

## 5. `index_col`: use a column as the row labels

In [ ]:
# enrollee_id becomes the index instead of 0, 1, 2, ...
pd.read_csv("data/aug_train.csv", index_col="enrollee_id").head(3)

## 6. `header`: choose which line holds the column names

In [ ]:
# Before: line 0 is only commas, so pandas invents names "Unnamed: 0", "Unnamed: 1", ...
pd.read_csv("data/test.csv")

In [ ]:
# After: header=1 says line 1 holds the column names
pd.read_csv("data/test.csv", header=1)

## 7. `usecols`: load only some columns

In [ ]:
pd.read_csv("data/aug_train.csv", usecols=["enrollee_id", "gender", "education_level"]).head(3)

## 8. One column as a Series (`squeeze` was removed)

In [ ]:
# Old code: pd.read_csv(..., usecols=["gender"], squeeze=True) -> TypeError in pandas 2 and 3
# Now: read a one-column DataFrame, then squeeze it into a Series
gender = pd.read_csv("data/aug_train.csv", usecols=["gender"]).squeeze("columns")
print(type(gender))
gender.head(3)

## 9. `skiprows` and `nrows`

In [ ]:
# skiprows counts FILE lines: line 0 is the header, lines 1 and 2 are the first two data rows
pd.read_csv("data/aug_train.csv", skiprows=[1, 2]).head(3)

In [ ]:
# careful: skipping line 0 throws away the header, so the first data row becomes the column names
pd.read_csv("data/aug_train.csv", skiprows=[0]).iloc[:2, :4]

In [ ]:
# a function decides line by line: here keep the header and every odd line
pd.read_csv("data/aug_train.csv", skiprows=lambda i: i > 0 and i % 2 == 0).shape

In [ ]:
# nrows: read only the first 100 data rows
pd.read_csv("data/aug_train.csv", nrows=100).shape

## 10. `encoding`

In [ ]:
# Before: this file is not UTF-8, so the default decoding fails
try:
    pd.read_csv("data/zomato.csv")
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e)

In [ ]:
# After: tell pandas the encoding
zomato = pd.read_csv("data/zomato.csv", encoding="latin-1")
print(zomato.shape)
zomato[["Restaurant Name", "City"]].iloc[20:27]   # loads, but some names look garbled (see the Note)

## 11. Bad lines: `on_bad_lines` (was `error_bad_lines`)

In [ ]:
# Before: line 5 has an extra ";" inside the title, so it has 5 fields instead of 4
try:
    pd.read_csv("data/books.csv", sep=";")
except pd.errors.ParserError as e:
    print("ParserError:", e)

In [ ]:
# After: skip the bad line and load the rest
# (old code used error_bad_lines=False, which no longer exists)
pd.read_csv("data/books.csv", sep=";", on_bad_lines="skip", dtype={"isbn": str})

In [ ]:
# "warn" also skips, but prints which line it dropped
pd.read_csv("data/books.csv", sep=";", on_bad_lines="warn").shape

## 12. `dtype`: choose a column's type

In [ ]:
# Before: target holds only 0.0 and 1.0, stored as float64 (8 bytes per value)
print(df["target"].dtype, df["target"].memory_usage(index=False), "bytes")

In [ ]:
# After: int8 needs 1 byte per value. (int64, what "int" gives, is still 8 bytes.)
small = pd.read_csv("data/aug_train.csv", dtype={"target": "int8"})
print(small["target"].dtype, small["target"].memory_usage(index=False), "bytes")

## 13. `parse_dates`: read dates as dates

In [ ]:
# Before: the date column is text
ipl = pd.read_csv("data/ipl_matches_2008_2020.csv")
ipl["date"].dtype

In [ ]:
# After: a real datetime column
ipl = pd.read_csv("data/ipl_matches_2008_2020.csv", parse_dates=["date"])
print(ipl["date"].dtype)

# date tools now work: year, weekday, differences
print(ipl["date"].dt.year.head(2).tolist(), ipl["date"].dt.day_name().head(2).tolist())
print("first to last match:", ipl["date"].max() - ipl["date"].min())

In [ ]:
# Day, month and year in separate columns? parse_dates can no longer combine them (removed in pandas 3).
# Read them normally, then build one date column with pd.to_datetime.
parts = pd.DataFrame({"year": [2008, 2008], "month": [4, 4], "day": [18, 19]})
parts["date"] = pd.to_datetime(parts[["year", "month", "day"]])
parts

## 14. `converters`: change a column while reading

In [ ]:
def rename(name):
    # shorten one team's name, leave the rest unchanged
    if name == "Royal Challengers Bangalore":
        return "RCB"
    return name

rename("Royal Challengers Bangalore")

In [ ]:
# the function runs on every value of team1 as the file is read
ipl = pd.read_csv("data/ipl_matches_2008_2020.csv", converters={"team1": rename})
ipl["team1"].value_counts().head(3)

## 15. `na_values`: extra values that mean "missing"

In [ ]:
# Before
pd.read_csv("data/aug_train.csv")["gender"].value_counts(dropna=False)

In [ ]:
# After: treat "Male" as missing (a demonstration only; real placeholders are things like "-" or "?")
pd.read_csv("data/aug_train.csv", na_values=["Male"])["gender"].value_counts(dropna=False)

## 16. `chunksize`: read a big file in pieces

In [ ]:
# chunksize returns a reader, not a DataFrame: nothing is loaded yet
reader = pd.read_csv("data/aug_train.csv", chunksize=300)

# each loop step loads one chunk (a DataFrame of up to 300 rows)
total = 0
for chunk in reader:
    print(chunk.shape)
    total += len(chunk)
print("rows processed:", total)